# Process Data & Train PyTorch CNN Model (Local Machine)
Xử lý dữ liệu và train model trên máy cá nhân

**Quy trình:**
1. Load raw data từ server (NetCDF)
2. Cloud masking → NDVI → Fill NaN → Monthly aggregation
3. Chuẩn bị training data + augmentation
4. Train PyTorch CNN model
5. Evaluate & save model

In [ ]:
%%time
import importlib
import sys
import os

# Import custom functions
import new_import_ODC

importlib.reload(new_import_ODC)
from new_import_ODC import *

# Setup matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

print("✅ Libraries imported successfully")

In [ ]:
## Load raw data from NetCDF files
print("📂 Loading raw satellite data from NetCDF files...\n")

data_dir = "data_for_training"

# Load Sentinel-2
print("📡 Loading Sentinel-2...")
s2_path = os.path.join(data_dir, "sentinel2_raw.nc")
data_s2 = xr.open_dataset(s2_path)
print(f"   ✅ Shape: {data_s2.dims}")
print(f"   Bands: {list(data_s2.data_vars.keys())}")

# Load Sentinel-1
print("\n📡 Loading Sentinel-1...")
s1_path = os.path.join(data_dir, "sentinel1_raw.nc")
data_s1 = xr.open_dataset(s1_path)
print(f"   ✅ Shape: {data_s1.dims}")
print(f"   Bands: {list(data_s1.data_vars.keys())}")

print("\n✅ All raw data loaded")

In [ ]:
%%time
## Process Sentinel-2: Cloud masking
print("☁️  Applying cloud mask (SCL band)...\n")

# Extract raw data
data = data_s2

# Apply cloud mask using SCL band
result = mask_clean(data)
print(f"✅ Cloud mask applied")
print(f"   Shape: {result.dims}")

# Compute to ensure data is loaded
result = result.compute()
print(f"✅ Data computed to memory")

In [ ]:
## Calculate NDVI from cloud-masked data
print("🌱 Calculating NDVI...\n")

ds1 = calculate_indices(result, index="NDVI", satellite_mission="s2")
ndvi = ds1["NDVI"]

print(f"✅ NDVI calculated")
print(f"   Shape: {ndvi.shape}")
print(f"   Value range: [{ndvi.min().values:.3f}, {ndvi.max().values:.3f}]")

In [ ]:
## Fill missing values with seasonal interpolation
print("🔧 Filling missing values (cloud pixels)...\n")

time_split = [
    slice("2022-09-01", "2023-01-01"),
    slice("2023-01-01", "2023-05-01"),
    slice("2023-05-01", "2023-07-01"),
    slice("2023-07-01", "2023-10-01"),
]

fill_nan_ndvi = fill_nan(ndvi, time_split)
print(f"✅ Missing values filled")
print(f"   NaN pixels remaining: {fill_nan_ndvi.isna().sum().values}")
print(f"   Valid pixels: {(~fill_nan_ndvi.isna()).sum().values}")

In [ ]:
%%time
## Monthly aggregation of NDVI and S1 data
print("📊 Aggregating to monthly averages...\n")

# NDVI monthly average
print("   - NDVI monthly...")
average_ndvi = fill_nan_ndvi.resample(time="1M").mean()
average_ndvi = average_ndvi.compute()

# S1 monthly average
print("   - Sentinel-1 VH/VV monthly...")
average_vh = calculate_average(data_s1['VH'], time_pattern='1M')
average_vv = calculate_average(data_s1['VV'], time_pattern='1M')

print(f"\n✅ Monthly aggregation complete")
print(f"   NDVI shape: {average_ndvi.shape}")
print(f"   VH shape: {average_vh.shape}")
print(f"   VV shape: {average_vv.shape}")

In [ ]:
## Load training data and prepare datasets
print("📋 Loading training data...\n")

train_shp_dir = os.path.join(data_dir, "train_data")
train_shp_path = os.path.join(train_shp_dir, "ST_training data_updated_1130points_new.shp")

print(f"   - Loading from {train_shp_path}...")
train = load_train_data(train_shp_path)
print(f"   ✅ Loaded {len(train)} training points")
print(f"   Classes: {train['Class'].unique()}")

# Define label mapping
label_mapping = {
    "Lua tom": "0",
    "Lua": "1",
    "CHN": "2",
    "CLN": "3",
    "TS": "4",
    "Song": "5",
    "Dat xay dung": "6",
    "Rung": "7",
}

print(f"\n   Label mapping: {label_mapping}")

# Prepare datasets (extract S2 + S1 values at training points)
print(f"\n   - Extracting features at training points...")
datasets = get_data_sen1_and_sen2(train, average_ndvi, average_vh, average_vv)

print(f"✅ Training dataset prepared")
print(f"   Features shape: {datasets[0].shape if hasattr(datasets[0], 'shape') else 'N/A'}")

In [ ]:
## Split training/validation/test data
print("✂️  Splitting data into train/validation/test...\n")

X_train, X_val, X_test, y_train, y_val, y_test = split_train_data(
    train, label_mapping, datasets
)

print(f"   Train set: {X_train.shape[0]} samples")
print(f"   Val set:   {X_val.shape[0]} samples")
print(f"   Test set:  {X_test.shape[0]} samples")

print(f"\n✅ Data split complete")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape: {X_val.shape}")
print(f"   X_test shape: {X_test.shape}")

In [ ]:
%%time
## Train PyTorch CNN Model
print("🤖 Training PyTorch CNN model...\n")

# Train the model using the function from new_import_ODC
model = train_cnn_pytorch(
    X_train, X_val, 
    y_train, y_val,
    epochs=50,
    batch_size=32,
    learning_rate=0.001,
    patience=10
)

print(f"\n✅ Model training complete")

In [ ]:
## Evaluate model on test set
print("📊 Evaluating model on test set...\n")

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import torch

# Get predictions on test set
device = torch.device('cpu')
model = model.to(device)
model.eval()

with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred_probs = model(X_test_tensor).cpu().numpy()
    y_pred_test = np.argmax(y_pred_probs, axis=1)

test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"✅ Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

print(f"\n📈 Classification Report:\n")
print(classification_report(y_test, y_pred_test, 
                          target_names=list(label_mapping.keys())))

print(f"\n🔲 Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
## Save trained model
print("💾 Saving trained model...\n")

model_path = "model_cnn_pytorch_local.pth"
torch.save(model.state_dict(), model_path)
print(f"   ✅ Model saved to {model_path}")

# Also save as PyTorch checkpoint with metadata
checkpoint = {
    'model_state_dict': model.state_dict(),
    'accuracy': test_accuracy,
    'label_mapping': label_mapping,
    'num_classes': len(label_mapping),
    'input_features': X_train.shape[1]
}

checkpoint_path = "model_cnn_pytorch_local_checkpoint.pth"
torch.save(checkpoint, checkpoint_path)
print(f"   ✅ Checkpoint saved to {checkpoint_path}")

print(f"\n✅ Model training & evaluation complete!")
print(f"   Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Model ready for prediction on full spatial extent")